1. put this notebook into Colab
2. add secrets. Hugging Face (HF_TOKEN) also recommended but not required
3. put the .gz file into your Google Drive in /erica/data
    - Colab will ask for Drive mount authorization  
4. when instance crashes, you can run the last block to resume rag insertion

In [ ]:
!pip install numpy==1.25.2 --upgrade

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!pip install nano-graphrag ollama nest-asyncio networkx sentence-transformers transformers --quiet

In [ ]:
# ===============================
# Data Ingestion: build / resume RAG graph
# ===============================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# Paths and working directories
# -------------------------------
DATA_DIR = "/content/drive/MyDrive/erica/data"
WORKING_DIR = "/content/drive/MyDrive/erica/nano_graphrag_cache_ollama"
MODEL = "qwen2"
BATCH_SIZE = 10  # save progress every 10 pages

# -------------------------------
# Imports
# -------------------------------
import os, gzip, json
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()
import logging
import ollama
import torch
from nano_graphrag import GraphRAG, QueryParam
from nano_graphrag._utils import wrap_embedding_func_with_attrs, compute_args_hash
from nano_graphrag.base import BaseKVStorage
from sentence_transformers import SentenceTransformer
from time import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("nano-graphrag")

# -------------------------------
# Device auto-detect
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# -------------------------------
# Embedding model
# -------------------------------
EMBED_MODEL = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    cache_folder=WORKING_DIR,
    device=device
)

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBED_MODEL.get_sentence_embedding_dimension(),
    max_token_size=EMBED_MODEL.max_seq_length,
)
async def local_embedding(texts: list[str]):
    return EMBED_MODEL.encode(texts, normalize_embeddings=True)

# -------------------------------
# Ollama wrapper
# -------------------------------
async def ollama_model_if_cache(prompt, system_prompt=None, history_messages=[], **kwargs):
    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    if system_prompt is None:
        system_prompt = """
        You are building a knowledge graph from the text. Follow this schema:

        NODES:
          - concept: {id, title, difficulty, aliases, definitions}
          - resource: {type ∈ {pdf, slide, video, web}, span, timecodes}
          - example: worked example snippets

        EDGES:
          - prereq_of(u → v)
          - explains(resource → concept)
          - exemplifies(example → concept)
          - near_transfer(concept ↔ concept)

        Return a JSON object with "nodes" and "edges" following this schema.
        """

    ollama_client = ollama.AsyncClient()
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
    args_hash = None
    if hashing_kv:
        args_hash = compute_args_hash(MODEL, messages)
        cached = await hashing_kv.get_by_id(args_hash)
        if cached is not None:
            return cached["return"]

    response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
    result = response["message"]["content"]

    if hashing_kv and args_hash:
        await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})

    return result

# -------------------------------
# Utility
# -------------------------------
def remove_if_exist(file):
    if os.path.exists(file):
        os.remove(file)

# -------------------------------
# Insert function: resume-safe
# -------------------------------
def insert(max_items_per_file: int = None, start_from_page: int = 0):
    """
    Index pages from JSON.GZ files.
    If max_items_per_file=None, process all pages in each file.
    Tracks progress in WORKING_DIR/progress.json
    """
    progress_file = Path(WORKING_DIR) / "progress.json"

    # Remove previous cache if starting fresh
    if start_from_page == 0:
        for f in [
            "vdb_entities.json",
            "kv_store_full_docs.json",
            "kv_store_text_chunks.json",
            "kv_store_community_reports.json",
            "graph_chunk_entity_relation.graphml",
        ]:
            remove_if_exist(Path(WORKING_DIR) / f)

    rag = GraphRAG(
        working_dir=WORKING_DIR,
        enable_llm_cache=True,
        best_model_func=ollama_model_if_cache,
        cheap_model_func=ollama_model_if_cache,
        embedding_func=local_embedding,
    )

    start_time = time()
    page_counter = start_from_page

    for file in Path(DATA_DIR).glob("*.json.gz"):
        print(f"Loading {file}")
        with gzip.open(file, "rt", encoding="utf-8") as f:
            obj = json.load(f)
        pages = obj if isinstance(obj, list) else [obj]

        # Determine how many pages to process
        pages_to_process = pages if max_items_per_file is None else pages[:max_items_per_file]

        for i, page in enumerate(pages_to_process):
            if page_counter < start_from_page:
                page_counter += 1
                continue

            text = page.get("text", "").strip()
            if not text:
                print(f"Skipped page {i} in {file}: no text")
                page_counter += 1
                continue

            print(f"Inserting page {i} from {file.name} | length: {len(text)}")
            rag.insert(text)

            page_counter += 1

            # Save progress after each page
            progress_file.write_text(json.dumps({"last_page": page_counter}))

            # Save after every BATCH_SIZE pages
            if page_counter % BATCH_SIZE == 0:
                print(f"--- Saving progress at page {page_counter} ---")
                rag.save_graph()   # saves graphml file
                rag.save_kv_cache()  # saves embeddings cache

    # Final save
    print(f"--- Finished inserting {page_counter} pages. Saving final progress ---")
    rag.save_graph()
    rag.save_kv_cache()
    print("Indexing time:", time() - start_time)

In [ ]:
import json
from pathlib import Path

progress_file = Path(WORKING_DIR) / "progress.json"
start_from = 0
if progress_file.exists():
    start_from = json.load(open(progress_file))["last_page"]

insert(max_items_per_file=None, start_from_page=start_from)  # process all remaining pages

FYI. test code that I shared the query result of.

Nano-graphRAG implementation using transformers and Ollama 
- only inserts a subset(5) of pages in the document
- query is hard coded

In [ ]:
# # Install required packages
# !pip install nano-graphrag ollama nest-asyncio networkx sentence-transformers transformers pyngrok --quiet

# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# DATA_DIR = "/content/drive/MyDrive/erica/data"

# # Start Ollama server
# import time
# !pkill -f ollama  # kill leftover Ollama processes
# !ollama serve > /dev/null 2>&1 &  # run in background quietly
# time.sleep(5)  # wait for server to initialize

# # Pull required models
# !ollama pull nomic-embed-text
# !ollama pull qwen2

# # ===============================
# # Import and set up nano-graphrag
# # ===============================
# import os, sys, gzip, json
# from pathlib import Path
# import nest_asyncio
# nest_asyncio.apply()
# sys.path.append("..")

# import logging
# import ollama
# import numpy as np
# from nano_graphrag import GraphRAG, QueryParam
# from nano_graphrag.base import BaseKVStorage
# from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs
# from sentence_transformers import SentenceTransformer

# logging.basicConfig(level=logging.WARNING)
# logging.getLogger("nano-graphrag").setLevel(logging.INFO)


# # Working directories and models
# WORKING_DIR = "./nano_graphrag_cache_ollama"
# MODEL = "qwen2"

# EMBED_MODEL = SentenceTransformer(
#     "sentence-transformers/all-MiniLM-L6-v2",
#     cache_folder=WORKING_DIR,
#     device="cpu"
# )

# @wrap_embedding_func_with_attrs(
#     embedding_dim=EMBED_MODEL.get_sentence_embedding_dimension(),
#     max_token_size=EMBED_MODEL.max_seq_length,
# )
# async def local_embedding(texts: list[str]) -> np.ndarray:
#     return EMBED_MODEL.encode(texts, normalize_embeddings=True)

# # -------------------------------
# # Ollama wrapper with caching
# # -------------------------------
# async def ollama_model_if_cache(prompt, system_prompt=None, history_messages=[], **kwargs) -> str:
#     kwargs.pop("max_tokens", None)
#     kwargs.pop("response_format", None)

#     if system_prompt is None:
#         system_prompt = """
#         You are building a knowledge graph from the text. Follow this schema:

#         NODES:
#           - concept: {id, title, difficulty, aliases, definitions}
#           - resource: {type ∈ {pdf, slide, video, web}, span, timecodes}
#           - example: worked example snippets

#         EDGES:
#           - prereq_of(u → v)
#           - explains(resource → concept)
#           - exemplifies(example → concept)
#           - near_transfer(concept ↔ concept)

#         Return a JSON object with "nodes" and "edges" following this schema.
#         """

#     ollama_client = ollama.AsyncClient()
#     messages = [{"role": "system", "content": system_prompt}]
#     messages.extend(history_messages)
#     messages.append({"role": "user", "content": prompt})

#     hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
#     args_hash = None
#     if hashing_kv:
#         args_hash = compute_args_hash(MODEL, messages)
#         cached = await hashing_kv.get_by_id(args_hash)
#         if cached is not None:
#             return cached["return"]

#     response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
#     result = response["message"]["content"]

#     if hashing_kv and args_hash:
#         await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})

#     return result

# # -------------------------------
# # Utility
# # -------------------------------
# def remove_if_exist(file):
#     if os.path.exists(file):
#         os.remove(file)

# # ===============================
# # Insert function: subset of pages
# # ===============================
# import random

# def insert(max_items_per_file: int = 5):
#     """
#     Index only a subset of pages from each file.
#     max_items_per_file: max number of pages to index per .json.gz file
#     """
#     from time import time

#     # Remove previous cache
#     for f in [
#         "vdb_entities.json",
#         "kv_store_full_docs.json",
#         "kv_store_text_chunks.json",
#         "kv_store_community_reports.json",
#         "graph_chunk_entity_relation.graphml",
#     ]:
#         remove_if_exist(Path(WORKING_DIR) / f)

#     rag = GraphRAG(
#         working_dir=WORKING_DIR,
#         enable_llm_cache=True,
#         best_model_func=ollama_model_if_cache,
#         cheap_model_func=ollama_model_if_cache,
#         embedding_func=local_embedding,
#     )

#     start = time()

#     # Load files from Google Drive
#     for file in Path(DATA_DIR).glob("*.json.gz"):
#         print(f"Loading {file}")
#         with gzip.open(file, "rt", encoding="utf-8") as f:
#             obj = json.load(f)

#         pages = obj if isinstance(obj, list) else [obj]

#         # Select a subset of pages
#         selected_pages = pages[:max_items_per_file]

#         for i, page in enumerate(selected_pages):
#             text = page.get("text", "").strip()
#             if not text:
#                 print(f"Skipped item {i} in {file}: no text")
#                 continue

#             print(f"Inserting page {i} from {file.name} | length: {len(text)}")
#             rag.insert(text)

#     print("Indexing time:", time() - start)

# # ===============================
# # Query function
# # ===============================
# def query():
#     rag = GraphRAG(
#         working_dir=WORKING_DIR,
#         best_model_func=ollama_model_if_cache,
#         cheap_model_func=ollama_model_if_cache,
#         embedding_func=local_embedding,
#     )
#     print(rag.query("What is this course all about?", param=QueryParam(mode="global")))

# # ===============================
# # Run insertion + query
# # ===============================
# insert(max_items_per_file=5)  # change this number to control subset size
# query()
